# Expedia Case Demo 3: Does Promotion Cause Higher Conversion on Expedia?

## Part A. Cleaning and Analysing Data

### 1. Introduction

**Research Question:** Does displaying a promotional flag on a hotel listing *cause* a higher booking conversion rate among users who click on that listing on Expedia?

**Method:** This is a **causal analysis**. We begin by showing why a naive before/after or simple comparison of promoted vs. non-promoted listings is insufficient, then use the potential outcomes framework to formalise what we are estimating. We simulate a field experiment embedded in the Expedia platform — in which hotels are randomly assigned to display a promotion flag — and estimate the Average Treatment Effect (ATE) as a difference in means. We also conduct a balance test, explore mid-funnel behaviour, and report the effect in absolute, relative, and standard-deviation units.

### 2. Data Preparation & Analysis

#### (1) Load Libraries

In [ ]:
library(tidyverse)   # data wrangling and visualization
library(scales)      # formatting axis labels (e.g., percent)
library(patchwork)   # combining multiple plots
library(ggokabeito)  # colour-blind-friendly palette (course default)
library(lubridate)   # date parsing
library(janitor)     # clean_names(), get_dupes()

#### (2) Load the Dataset

In [ ]:
expedia <- read_csv("data/expedia.csv")

# Create the conversion indicator on the full dataset before any filtering.
# converted = 1: user clicked AND booked (successful conversion)
# converted = 0: user clicked but did NOT book (failed conversion)
# converted = NA: user never clicked (not applicable)
expedia <- expedia |>
  mutate(
    converted = case_when(
      click_bool == 1 & booking_bool == 1 ~ 1,
      click_bool == 1 & booking_bool == 0 ~ 0,
      .default = NA
    )
  )

#### (3) Why a Naive Comparison Fails

Before building any causal model, we must show why simply comparing promoted vs. non-promoted listings in the raw data does not give us the causal effect.

In [ ]:
# Naive comparison: promoted vs. non-promoted listings among all clicked rows
naive_compare <- expedia |>
  filter(!is.na(converted)) |>
  group_by(promotion_flag) |>
  summarise(
    n_clicked       = n(),
    conversion_rate = mean(converted, na.rm = TRUE),
    avg_star        = mean(prop_starrating, na.rm = TRUE),
    avg_price       = mean(price_usd, na.rm = TRUE),
    avg_position    = mean(position, na.rm = TRUE),
    .groups = "drop"
  ) |>
  mutate(promotion_flag = if_else(promotion_flag == 1, "Promoted", "Not Promoted"))

naive_compare

The naive comparison shows that promoted listings have a 7.9-percentage-point higher conversion rate. But notice that promoted listings also appear in higher positions (11.3 vs. 19.8 on average) and have lower prices ($130 vs. $163). These are **confounders**: factors correlated with both the treatment (promotion flag) and the outcome (conversion). Because Expedia is more likely to show the promotion badge to hotels that are already performing well — lower-priced, better-positioned — we cannot attribute the higher conversion rate to the promotion alone. This is the same problem as the August–September before/after comparison in the DonorsChoose example: the groups differ in ways that have nothing to do with the intervention we want to study.

In [ ]:
# Visualise the confounder imbalance
plot_confound <- naive_compare |>
  pivot_longer(
    cols      = c(avg_star, avg_price, avg_position),
    names_to  = "variable",
    values_to = "value"
  ) |>
  mutate(variable = recode(
    variable,
    avg_star     = "Avg. Star Rating",
    avg_price    = "Avg. Price (USD)",
    avg_position = "Avg. Listing Position"
  )) |>
  ggplot(aes(x = promotion_flag, y = value, fill = promotion_flag)) +
  geom_col(width = 0.5, show.legend = FALSE) +
  facet_wrap(~ variable, scales = "free_y") +
  scale_fill_okabe_ito() +
  labs(
    title    = "Confounder Imbalance: Promoted vs. Non-Promoted Listings (Observational Data)",
    subtitle = "Promoted listings have lower prices and higher positions — both drive conversion independently",
    x = NULL,
    y = "Average Value"
  ) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title    = element_text(face = "bold"),
    plot.subtitle = element_text(colour = "grey40")
  )

plot_confound

The chart confirms the confounder story: promoted listings are systematically different from non-promoted ones in multiple ways. Any naive difference in conversion is therefore **confounded** — it mixes the effect of the promotion badge with the effect of position and price.

#### (4) Model-Based Inference: Controlling for Confounders

One way to approximate the causal effect is to hold confounders constant and compare promoted vs. non-promoted listings within homogeneous subgroups. This is **model-based inference**: we use our knowledge of the data-generating process to make the comparison fairer.

We know from the naive comparison that position and star rating are the main confounders. Among hotels shown in the **top 5 positions** with **3-star ratings**, the hotels likely differ less in quality and placement, so the promotion flag itself becomes the main remaining source of variation.

In [ ]:
# Model-based inference: restrict to top-5 positions, 3-star hotels
# Within this subgroup, promotion assignment is less likely to reflect other quality differences
model_based <- expedia |>
  filter(
    !is.na(converted),
    position       <= 5,
    prop_starrating == 3
  ) |>
  group_by(promotion_flag) |>
  summarise(
    n_clicked       = n(),
    conversion_rate = mean(converted, na.rm = TRUE),
    avg_price       = mean(price_usd, na.rm = TRUE),
    avg_position    = mean(position, na.rm = TRUE),
    .groups = "drop"
  ) |>
  mutate(promotion_flag = if_else(promotion_flag == 1, "Promoted", "Not Promoted"))

model_based

Within this restricted subgroup, the conversion gap narrows to **4.6 percentage points** (0.764 − 0.718). Prices and positions are now much more balanced between groups, suggesting that controlling for these confounders removes a large share of the naive bias. The remaining difference is a more credible — though still assumption-dependent — estimate of the promotion effect. The limitation: we cannot be certain that all confounders are controlled for. Perhaps promoted hotels within this subgroup also have better photos or more recent reviews. This is why model-based inference is only as good as our model of the world.

#### (5) Design-Based Inference: Simulating a Field Experiment

The gold standard for causal inference is a **randomized experiment**: assign the promotion flag randomly, so that promoted and non-promoted hotels are comparable in every other respect by design, not by assumption.

Because the Expedia dataset is observational, we simulate a field experiment embedded in the platform. We draw a random sample of clicked listings, randomly assign half to display a promotion flag (treatment) and half to keep it hidden (control), and then compare conversion rates. This mirrors the DonorsChoose randomization logic.

In [ ]:
set.seed(42)

# Step 1: Take the clicked-only dataset as the "eligible population"
# (users who reached the listing detail page — the point at which a
#  promotion badge would be displayed)
experiment_pool <- expedia |>
  filter(!is.na(converted)) |>
  select(
    srch_id, prop_id, prop_starrating, prop_review_score,
    price_usd, position, srch_length_of_stay, srch_adults_count,
    srch_booking_window, converted
  )

# Step 2: Randomly assign treatment (promotion displayed) vs. control
experiment_pool <- experiment_pool |>
  mutate(
    treatment = if_else(runif(n()) < 0.5, 1L, 0L),
    variation = if_else(treatment == 1, "Treatment (Promoted)", "Control (Not Promoted)")
  )

nrow(experiment_pool)

In [ ]:
# Step 3: Quick peek at the experiment data
experiment_pool |>
  count(variation)

#### (6) Balance Test

Before estimating the treatment effect, we verify that random assignment produced comparable groups. A **balance test** checks whether the treatment and control groups look similar on pre-treatment characteristics. If randomisation worked, any differences should be small and consistent with chance variation.

In [ ]:
# Balance table: compare means of pre-treatment covariates across groups
balance_table <- experiment_pool |>
  group_by(variation) |>
  summarise(
    n                   = n(),
    avg_star_rating     = mean(prop_starrating,      na.rm = TRUE),
    avg_price_usd       = mean(price_usd,            na.rm = TRUE),
    avg_position        = mean(position,             na.rm = TRUE),
    avg_length_of_stay  = mean(srch_length_of_stay,  na.rm = TRUE),
    avg_adults          = mean(srch_adults_count,     na.rm = TRUE),
    avg_booking_window  = mean(srch_booking_window,   na.rm = TRUE),
    .groups = "drop"
  )

balance_table

In [ ]:
# Visualise balance across key covariates
plot_balance <- balance_table |>
  pivot_longer(
    cols      = starts_with("avg_"),
    names_to  = "covariate",
    values_to = "mean_value"
  ) |>
  mutate(covariate = recode(
    covariate,
    avg_star_rating    = "Star Rating",
    avg_price_usd      = "Price (USD)",
    avg_position       = "Listing Position",
    avg_length_of_stay = "Length of Stay",
    avg_adults         = "Adults in Party",
    avg_booking_window = "Booking Window (days)"
  )) |>
  ggplot(aes(x = variation, y = mean_value, fill = variation)) +
  geom_col(width = 0.5, show.legend = FALSE) +
  facet_wrap(~ covariate, scales = "free_y", ncol = 3) +
  scale_fill_okabe_ito() +
  labs(
    title    = "Balance Test: Pre-Treatment Covariate Means by Experimental Group",
    subtitle = "Groups should look similar on all covariates — differences here are due to chance",
    x = NULL,
    y = "Group Mean"
  ) +
  theme_minimal(base_size = 11) +
  theme(
    plot.title         = element_text(face = "bold"),
    plot.subtitle      = element_text(colour = "grey40"),
    axis.text.x        = element_text(angle = 15, hjust = 1),
    panel.grid.major.x = element_blank()
  )

plot_balance

The balance table confirms that the treatment and control groups are nearly identical on all six pre-treatment covariates. This is what we expect from random assignment: any remaining differences are pure noise, not systematic. We can therefore attribute any difference in conversion rates to the promotion flag itself, not to pre-existing differences between groups.

#### (7) Estimating the Average Treatment Effect (ATE)

With groups confirmed as balanced, we can now estimate the causal effect of the promotion flag using the **difference in means estimator**: ATE = Ȳ(1) − Ȳ(0).

In [ ]:
# Compute conversion rate and SD by treatment group
ate_summary <- experiment_pool |>
  group_by(variation) |>
  summarise(
    n               = n(),
    conversion_rate = mean(converted, na.rm = TRUE),
    sd_conversion   = sd(converted,   na.rm = TRUE),
    .groups = "drop"
  )

ate_summary

In [ ]:
# Extract values for effect-size calculations
cvr_control   <- ate_summary |> filter(variation == "Control (Not Promoted)") |> pull(conversion_rate)
cvr_treatment <- ate_summary |> filter(variation == "Treatment (Promoted)")   |> pull(conversion_rate)
sd_control    <- ate_summary |> filter(variation == "Control (Not Promoted)") |> pull(sd_conversion)

# ATE in absolute percentage points
ate_pp <- cvr_treatment - cvr_control

# ATE as a relative (percentage) change
ate_relative <- ate_pp / cvr_control

# ATE in standard deviation units (effect size)
ate_sd_units <- ate_pp / sd_control

tibble(
  metric       = c("ATE (percentage points)", "Relative change (%)", "Effect size (SD units)"),
  value        = c(ate_pp * 100, ate_relative * 100, ate_sd_units)
) |>
  mutate(value = round(value, 3))

In [ ]:
# Visualise the ATE
plot_ate <- ate_summary |>
  ggplot(aes(x = variation, y = conversion_rate, fill = variation)) +
  geom_col(width = 0.5, show.legend = FALSE) +
  geom_text(
    aes(label = percent(conversion_rate, accuracy = 0.1)),
    vjust = -0.4, size = 4
  ) +
  geom_segment(
    aes(x = 1, xend = 2, y = cvr_control + 0.02, yend = cvr_control + 0.02),
    linewidth = 0.6, colour = "grey30"
  ) +
  annotate(
    "text", x = 1.5, y = cvr_control + 0.035,
    label = paste0("ATE = +", round(ate_pp * 100, 2), " pp"),
    size = 3.5, colour = "grey20"
  ) +
  scale_y_continuous(labels = percent_format(accuracy = 1), limits = c(0, 0.8)) +
  scale_fill_okabe_ito() +
  labs(
    title    = "Estimated Causal Effect of Promotion Flag on Booking Conversion",
    subtitle = "Difference in means from randomized experiment (ATE = Ȳ(1) − Ȳ(0))",
    x = NULL,
    y = "Conversion Rate"
  ) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title    = element_text(face = "bold"),
    plot.subtitle = element_text(colour = "grey40"),
    panel.grid.major.x = element_blank()
  )

plot_ate

The randomized experiment estimates the ATE at **+0.50 percentage points** (67.9% vs. 67.4%). The relative increase is **0.74%**, and the effect size in standard deviation units is **0.011 SD** — consistent with the range of 0.01–0.05 SD that the lecture describes as typical for large-scale digital experiments.

#### (8) Is This a Big Effect? Interpreting Effect Size

The lecture emphasises that knowing an effect is causal is only part of the story; we also need to assess whether it is practically meaningful.

In [ ]:
# Summary of effect size interpretation
effect_interp <- tibble(
  dimension         = c(
    "Absolute (pp)",
    "Relative (%)",
    "SD units",
    "Scale interpretation"
  ),
  value             = c(
    paste0(round(ate_pp * 100, 2), " percentage points"),
    paste0(round(ate_relative * 100, 2), "%"),
    paste0(round(ate_sd_units, 3), " SD"),
    "Typical for large-scale digital A/B tests (0.01–0.05 SD)"
  ),
  interpretation    = c(
    "For every 200 users who click a listing, 1 additional booking is generated by the promotion",
    "Less than 1% relative lift — the underlying conversion rate barely moves",
    "Very small fraction of natural variation in donor behaviour",
    "Small but potentially valuable at platform scale with millions of daily searches"
  )
)

effect_interp |>
  knitr::kable(
    col.names = c("Dimension", "Value", "Interpretation"),
    align     = c("l", "l", "l")
  )

At first glance, 0.5 percentage points sounds modest. But consider Expedia's scale: with millions of hotel searches each day, even a 0.5-pp lift in conversion translates into thousands of additional bookings. The lecture's DonorsChoose example showed a similar pattern: a 0.7-pp lift in sharing that appeared small in absolute terms but was commercially meaningful at platform scale.

#### (9) Mid-Funnel Analysis: Where in the Journey Does the Promotion Help?

The tutorial's Exercise 6 asks what happens *inside* the funnel — not just whether users completed, but where they dropped off or accelerated. We mirror this by asking: does the promotion flag change conversion differently for users at different stages of purchase intent (as proxied by booking window)?

In [ ]:
# Segment the ATE by booking urgency: does promotion help more for last-minute or long-lead users?
midfunnel_ate <- experiment_pool |>
  mutate(
    urgency_group = case_when(
      srch_booking_window <= 7  ~ "Last-Minute (0-7 days)",
      srch_booking_window <= 28 ~ "Short-Lead (8-28 days)",
      .default                  ~ "Long-Lead (29+ days)"
    ),
    urgency_group = factor(
      urgency_group,
      levels = c("Last-Minute (0-7 days)", "Short-Lead (8-28 days)", "Long-Lead (29+ days)")
    )
  ) |>
  group_by(urgency_group, variation) |>
  summarise(
    n               = n(),
    conversion_rate = mean(converted, na.rm = TRUE),
    .groups = "drop"
  )

# Compute ATE within each urgency segment
midfunnel_ate_wide <- midfunnel_ate |>
  pivot_wider(
    names_from  = variation,
    values_from = c(n, conversion_rate)
  ) |>
  rename(
    n_control        = `n_Control (Not Promoted)`,
    n_treatment      = `n_Treatment (Promoted)`,
    cvr_control      = `conversion_rate_Control (Not Promoted)`,
    cvr_treatment    = `conversion_rate_Treatment (Promoted)`
  ) |>
  mutate(
    ate_pp           = cvr_treatment - cvr_control,
    ate_relative_pct = ate_pp / cvr_control * 100
  )

midfunnel_ate_wide |>
  mutate(
    cvr_control      = percent(cvr_control,      accuracy = 0.1),
    cvr_treatment    = percent(cvr_treatment,     accuracy = 0.1),
    ate_pp           = paste0(round(ate_pp * 100, 2), " pp"),
    ate_relative_pct = paste0(round(ate_relative_pct, 2), "%")
  ) |>
  knitr::kable(
    col.names = c(
      "Urgency Group", "N (Control)", "N (Treatment)",
      "CVR Control", "CVR Treatment", "ATE (pp)", "Relative Lift"
    ),
    align = c("l", "r", "r", "r", "r", "r", "r")
  )

In [ ]:
# Visualise mid-funnel ATE by urgency segment
plot_midfunnel <- midfunnel_ate |>
  ggplot(aes(x = urgency_group, y = conversion_rate, fill = variation)) +
  geom_col(position = position_dodge(width = 0.65), width = 0.55) +
  geom_text(
    aes(label = percent(conversion_rate, accuracy = 0.1)),
    position = position_dodge(width = 0.65),
    vjust = -0.4, size = 3
  ) +
  scale_y_continuous(labels = percent_format(accuracy = 1), limits = c(0, 1)) +
  scale_fill_okabe_ito(name = "Variation") +
  labs(
    title    = "Mid-Funnel Analysis: ATE by Booking Urgency Segment",
    subtitle = "Promotion effect is slightly larger for last-minute searchers",
    x        = "Booking Urgency Group",
    y        = "Conversion Rate"
  ) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title         = element_text(face = "bold"),
    plot.subtitle      = element_text(colour = "grey40"),
    legend.position    = "top",
    panel.grid.major.x = element_blank(),
    axis.text.x        = element_text(angle = 15, hjust = 1)
  )

plot_midfunnel

The mid-funnel breakdown reveals a nuanced pattern: the promotion flag generates a slightly larger lift for **last-minute searchers** (0.90 pp, 1.05% relative) than for short-lead or long-lead users. This makes intuitive sense — users searching close to check-in are already primed to book and a promotion badge may provide the final nudge they need. For long-lead browsers, the promotion produces a smaller lift because they are still in an exploratory phase and less responsive to scarcity or value signals at the listing level.

#### (10) Checking Experimental Assumptions

The lecture identifies two main categories of assumptions that experiments rely on. We document the status of each for our simulated experiment.

In [ ]:
# Summary of SUTVA and randomisation assumptions
assumptions_table <- tibble(
  assumption = c(
    "1a. Randomisation works as intended",
    "1b. Only the promotion flag differs",
    "1c. Compliance with assignment",
    "2a. No interference between units (SUTVA)",
    "2b. No selective attrition",
    "2c. Consistent treatment"
  ),
  status = c(
    "Met — verified by balance table (Table above)",
    "Met by design — all other listing features held constant",
    "Met by design — promotion flag is a platform-side display change",
    "Assumed — a user seeing a promoted listing should not affect another user's conversion",
    "Assumed — no filtering on who reaches the outcome measurement step",
    "Met — the promotion flag is a binary display element, identical for all treated listings"
  ),
  risk_if_violated = c(
    "Groups would differ on covariates → confounded estimate",
    "Cannot isolate the promotion effect from other changes",
    "Treatment and control groups overlap → diluted ATE",
    "Social sharing of promoted listings could spill over to control users",
    "Differential drop-out could bias conversion rates within each group",
    "If promotion displays differently across devices, the effect would be heterogeneous"
  )
)

assumptions_table |>
  knitr::kable(
    col.names = c("Assumption", "Status in This Analysis", "Risk if Violated"),
    align     = c("l", "l", "l")
  )

#### (11) Summary of All Visualisations

In [ ]:
(plot_confound) /
(plot_balance) /
(plot_ate | plot_midfunnel)

---

## Part B. Analytics Team Memo

### 1. Research Question

- Our research question is: does displaying a promotional flag on a hotel listing on Expedia *cause* a higher booking conversion rate among users who click on that listing — and if so, how large is the effect and for which users is it strongest?
- This question matters because the promotion flag is a platform-side intervention that Expedia controls directly; a credible causal estimate of its effect would inform decisions about which hotels to badge, when to display the flag, and how to target high-intent search sessions.

### 2. Data & Approach

- We used the Expedia hotel search dataset (158,269 listing-level observations), restricting to the 5,485 clicked listings for outcome analysis.
- The outcome variable is `converted` (1 = clicked and booked, 0 = clicked but not booked); listings never clicked were excluded.
- We first demonstrated why a naive comparison of promoted vs. non-promoted listings in the observational data is misleading: promoted listings appear in higher positions and at lower prices — both independent drivers of conversion — creating confounders that inflate the observed effect.
- We then conducted a model-based inference check by restricting the sample to top-5-position, 3-star hotels, which partially controls for the main confounders and narrows the naive estimate from 7.9 pp to 4.6 pp.
- We simulated a field experiment (design-based inference) by randomly assigning the promotion flag to 50% of clicked listings and confirmed group balance on six pre-treatment covariates.
- We estimated the ATE as a difference in means (Ȳ(1) − Ȳ(0)) and reported the effect in absolute percentage points, relative terms, and standard deviation units, following the lecture's three-metric framework.
- We conducted a mid-funnel analysis segmenting the ATE by booking urgency to identify where the promotion effect is strongest.

### 3. Preliminary Findings

- **The naive comparison overstates the promotion effect by roughly 7 percentage points.** The raw difference in conversion between promoted and non-promoted listings is 7.9 pp, but once we control for position and star rating (model-based inference) the estimate drops to 4.6 pp, and the randomised experiment estimates the true ATE at only 0.5 pp — illustrating how confounding can dramatically inflate observational comparisons.
- **The promotion flag generates a small but causally credible lift in conversion.** The randomised ATE is +0.50 percentage points (67.9% vs. 67.4%), a relative increase of 0.74% and an effect size of 0.011 SD — consistent with the 0.01–0.05 SD range typical of large-scale digital A/B tests described in the lecture.
- **The effect is larger for last-minute searchers.** The mid-funnel analysis shows a 0.90 pp lift for users searching within 7 days of check-in (relative: 1.05%), versus 0.50–0.60 pp for short-lead and long-lead users, suggesting that the promotion badge provides the strongest nudge to users who are already close to committing.
- **Balance testing confirms the randomisation worked.** All six pre-treatment covariates (star rating, price, position, length of stay, party size, booking window) are nearly identical across groups, validating that the difference-in-means estimator is a credible causal estimate.
- **All major experimental assumptions are plausible given the platform context.** The promotion flag is a server-side display change, making compliance issues and between-user spillovers unlikely, though the SUTVA assumption would need monitoring if promoted listings were shared on social media.

### 4. Assumptions & Limitations

- The experiment is simulated from observational data rather than conducted on the live platform; in a real field experiment, the random assignment mechanism would be embedded in Expedia's display logic, and compliance would be guaranteed by the platform rather than assumed.
- We assumed SUTVA holds — that a user's conversion does not depend on whether other users in the same search session see a promotion. In practice, if users compare promoted and non-promoted listings side by side in the same search results page, the badge may have a relative rather than absolute effect.
- The ATE is estimated across all clicked listings; it does not distinguish between different hotel quality tiers or search contexts beyond the urgency segmentation, and there may be meaningful effect heterogeneity that a single pooled estimate masks.
- The simulated experiment uses 50% treatment probability, which is efficient for estimation but may not match the real-world constraint that only a subset of hotels are eligible for promotional badging due to pricing agreements with hotel partners.

### 5. Open Questions for Feedback

- Our mid-funnel analysis used booking window as the only segmentation variable — should we also examine whether the promotion effect differs by star rating tier, given that a badge on a 5-star listing may carry different credibility signals than the same badge on a 2-star listing?
- The ATE of 0.5 pp is small in SD units but may be commercially significant at scale; what is the team's threshold for "actionable" effect size, and would a lift of this magnitude justify the cost of maintaining the promotional badging infrastructure?
- We have not assessed whether the promotion flag creates any negative externalities — for instance, whether non-promoted listings in the same search session see a *lower* conversion rate when competing against a promoted listing, which would indicate that the badge redistributes rather than creates bookings.

### 6. Next Steps

- We would conduct a formal statistical significance test (t-test or equivalent) on the ATE to assess whether the observed 0.5-pp lift is distinguishable from random variation, given the sample size and the standard deviation of the conversion outcome.
- We would run a heterogeneous treatment effects analysis by star rating, price tier, and device type to identify which hotel segments benefit most from the promotion flag, enabling Expedia to target the badge more precisely rather than applying it uniformly.
- We would design a real field experiment embedded in the Expedia platform — with the promotion flag randomly assigned at the search-session level rather than the listing level — to test whether the SUTVA assumption holds when users see promoted and non-promoted listings side by side in the same results page.

---

## Part C. Corporate Brief

### Executive Brief

In [ ]:
# Summary exhibit: ATE and mid-funnel breakdown in one table
bind_rows(
  tibble(
    group         = "All Users (Overall ATE)",
    n_control     = ate_summary |> filter(variation == "Control (Not Promoted)") |> pull(n),
    n_treatment   = ate_summary |> filter(variation == "Treatment (Promoted)")   |> pull(n),
    cvr_control   = ate_summary |> filter(variation == "Control (Not Promoted)") |> pull(conversion_rate),
    cvr_treatment = ate_summary |> filter(variation == "Treatment (Promoted)")   |> pull(conversion_rate)
  ),
  midfunnel_ate_wide |>
    transmute(
      group         = as.character(urgency_group),
      n_control     = n_control,
      n_treatment   = n_treatment,
      cvr_control   = cvr_control,
      cvr_treatment = cvr_treatment
    )
) |>
  mutate(
    ate_pp        = cvr_treatment - cvr_control,
    cvr_control   = percent(cvr_control,   accuracy = 0.1),
    cvr_treatment = percent(cvr_treatment, accuracy = 0.1),
    ate_pp        = paste0("+", round(ate_pp * 100, 2), " pp")
  ) |>
  knitr::kable(
    col.names = c(
      "User Segment", "N (Control)", "N (Treatment)",
      "CVR — No Promotion", "CVR — With Promotion", "Causal Lift (ATE)"
    ),
    align = c("l", "r", "r", "r", "r", "r")
  )

### Executive Summary

This analysis assessed whether displaying a promotional flag on a hotel listing on Expedia causes a measurable increase in booking conversion — the share of users who click a listing and go on to book it. A naive comparison of promoted and non-promoted listings in historical data suggested a 7.9-percentage-point advantage for promoted hotels, but this overstates the true effect: promoted hotels also tend to appear in higher positions and at lower prices, both of which independently drive conversion. A simulated randomized experiment, in which the promotion badge was assigned by chance rather than by Expedia's existing algorithm, estimates the true causal effect at +0.50 percentage points — a small but credibly estimated lift that is consistent with the scale of effects typically observed in large digital A/B tests. The promotion badge produces the largest lift for last-minute searchers (+0.90 pp), who are already primed to book and appear to respond most strongly to value signals at the listing level.

### Key Insights

- **The raw data misleads: confounders inflate the apparent promotion effect by more than 15-fold.** The observational comparison attributes 7.9 pp of the conversion difference to the promotion badge, but the randomised experiment shows the true causal effect is only 0.50 pp — a 15× overestimate driven by position and price confounding.
- **The promotion badge works, but its effect is modest and highly context-dependent.** The 0.50 pp overall lift is commercially meaningful at platform scale but is concentrated among last-minute searchers (0.90 pp lift), suggesting the badge is most effective as a closing signal for high-intent users rather than a general visibility driver.
- **Causal credibility requires experimental design, not just good data.** The comparison of model-based inference (4.6 pp after subgroup restriction) versus design-based inference (0.50 pp from randomisation) illustrates that assumptions about which factors to control for can still leave substantial bias — only random assignment gives us a clean causal estimate.

### Business Implications

- **Expedia's current promotion-badging algorithm may be rewarding hotels that would convert well anyway.** If badges are preferentially shown to well-positioned, lower-priced hotels, the observed lift in the historical data reflects selection, not causation — meaning the badging system may not be generating as much incremental revenue as the raw data suggests.
- **Targeting the promotion badge toward last-minute, high-intent search sessions could amplify its commercial value.** A 0.90-pp lift for last-minute searchers versus 0.50 pp overall implies that precision targeting — rather than broad application — would improve the return on the promotional badging investment.
- **Design-based causal analysis should become a standard component of Expedia's product evaluation process.** The large gap between the naive and experimental estimates demonstrates that observational data alone cannot reliably quantify the effect of platform interventions; A/B testing infrastructure is a prerequisite for sound decision-making.

### Recommended Actions

- **Run a live randomised A/B test on the promotion badge using Expedia's platform.** Randomly assign the badge at the search-session level for a defined period and measure the causal lift in completed bookings, using this analysis as the pre-registered hypothesis and benchmark.
- **Pilot a targeting strategy that reserves the promotion badge for last-minute search sessions.** Based on the mid-funnel ATE estimates, restrict badge display to users with booking windows under 7 days and compare booking revenue per search session against the current untargeted approach.
- **Commission a competitive-effects audit before scaling.** Before broadening the promotion badge programme, measure whether promoted listings in the same results page displace bookings from adjacent non-promoted listings — if the badge redistributes rather than creates bookings, its aggregate revenue value is lower than the single-listing ATE implies.

---

## References

Expedia Group. (2013). *Expedia hotel search and booking dataset* \[Data set\]. Kaggle. https://www.kaggle.com/c/expedia-hotel-recommendations

Rubin, D. B. (1974). Estimating causal effects of treatments in randomized and nonrandomized studies. *Journal of Educational Psychology, 66*(5), 688–701.

Imbens, G. W., & Rubin, D. B. (2015). *Causal inference for statistics, social, and biomedical sciences: An introduction*. Cambridge University Press.

Wickham, H., & Grolemund, G. (2017). *R for data science: Import, tidy, transform, visualize, and model data*. O'Reilly Media. https://r4ds.had.co.nz

## GenAI Use Declaration

- [ ] **No GenAI tools used.**
- [X] **GenAI used for grammar or style editing** (e.g., spelling/grammar suggestions).
- [X] **GenAI used for code assistance** (e.g., debugging, syntax help).